# Quantum CNN for Intrusion Detection on MIoT Dataset

## Prerequisites & Setup

### Required File Structure
```
NCKH-hybrid-Quantum-Computing-and-GNNTechniques-for-IoT-IDS-in-5G-Edge-Computing/
├── Code/
│   └── Quantum_CNN_for_Intrusion_Detection_On_MIot_dataset.ipynb  (this file)
├── Dataset/
│   ├── Attack.csv
│   ├── environmentMonitoring.csv
│   └── patientMonitoring.csv
├── README.md
└── INTRODUCE.md
```

### Quick Start
1. **Ensure CSV files are in the `Dataset/` folder** (sibling of `Code/` folder)
2. Run cells sequentially from top to bottom
3. The notebook will auto-detect your environment (Local or Google Colab)

### Troubleshooting
- **"✗ Not found" messages**: Move CSV files to `../Dataset/` relative to this notebook
- **Google Colab users**: CSV files should be in Google Drive at `MyDrive/Dataset_v/`

In [ ]:
#Load & Merge Attack and Sensor Data
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import sys
import glob

# Detect environment - More reliable detection for Google Colab
def is_google_colab():
    """Check if running in Google Colab"""
    try:
        import google.colab
        return True
    except ImportError:
        return False

IS_COLAB = is_google_colab()
print(f"Running on {'Google Colab' if IS_COLAB else 'Local Machine'}")

csv_files = None
output_path = None

if IS_COLAB:
    # Google Colab setup
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        
        csv_files = [
            '/content/drive/MyDrive/Dataset_v/Bản sao của Attack.csv',
            '/content/drive/MyDrive/Dataset_v/Bản sao của environmentMonitoring.csv',
            '/content/drive/MyDrive/Dataset_v/Bản sao của patientMonitoring.csv'
        ]
        output_path = '/content/drive/MyDrive/Dataset_v/Bản sao của shuffled_data.csv'
        print("✓ Google Drive mounted successfully")
    except Exception as e:
        print(f"✗ Error mounting Google Drive: {e}")
        print("  Falling back to local dataset search...")
        IS_COLAB = False

# If Colab failed or running locally, search for local dataset
if not IS_COLAB or csv_files is None:
    print("\n" + "="*60)
    print("SETUP: Searching for local dataset files...")
    print("="*60)
    
    possible_paths = [
        '../Dataset',           # Standard location
        '../../Dataset',        # Parent directory
        '../dataset',           # Lowercase
        '../../dataset',
        'Dataset',              # Current directory
        'dataset',
    ]
    
    dataset_dir = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.isdir(path):
            csv_count = len(glob.glob(os.path.join(path, '*.csv')))
            if csv_count > 0:
                dataset_dir = os.path.abspath(path)
                print(f"✓ Found {csv_count} CSV files in: {dataset_dir}")
                break
    
    if dataset_dir is None:
        print("\n✗ ERROR: Cannot find Dataset folder!")
        print("\nExpected structure:")
        print("  Project/")
        print("  ├── Code/")
        print("  │   └── Quantum_CNN_for_Intrusion_Detection_On_MIot_dataset.ipynb")
        print("  └── Dataset/")
        print("      ├── Attack.csv")
        print("      ├── environmentMonitoring.csv")
        print("      └── patientMonitoring.csv")
        print("\nPlease ensure your CSV files are in the Dataset folder!")
        raise FileNotFoundError("Dataset folder not found. Please check file structure.")
    
    # Use glob to find all CSV files
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, '*.csv')))
    output_path = os.path.join(dataset_dir, 'shuffled_data.csv')
    
    print(f"\nCSV files found in {dataset_dir}:")
    for i, file in enumerate(csv_files, 1):
        file_size = os.path.getsize(file) / (1024*1024)  # Convert to MB
        print(f"  {i}. {os.path.basename(file)} ({file_size:.2f} MB)")
    print("\n" + "="*60)

# Verify csv_files is set
if csv_files is None or len(csv_files) == 0:
    raise RuntimeError("Failed to locate CSV files. Please check your dataset folder.")

print(f"\nDataset directory: {os.path.dirname(csv_files[0])}")

# Load and concatenate all datasets
try:
    dfs = [pd.read_csv(file, low_memory=False) for file in csv_files]
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"✓ Successfully loaded {len(dfs)} datasets")
    print(f"  Combined dataset shape: {combined_df.shape}")
except FileNotFoundError as e:
    print(f"✗ Error: {e}")
    print(f"  Please ensure dataset files are in the correct location.")
    raise
except Exception as e:
    print(f"✗ Unexpected error loading files: {e}")
    raise

# Shuffle dataset to prevent ordering bias
shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
shuffled_df.to_csv(output_path, index=False)
print(f"✓ Shuffled dataset saved to: {output_path}")

# Load shuffled dataset
df = pd.read_csv(output_path)
print(f"✓ Loaded shuffled dataset with shape: {df.shape}")

# Extract labels and remove categorical columns
y = df["label"]
df.drop(["class", "label"], axis=1, inplace=True)

# Convert categorical and object columns to numeric (handling mixed types)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values
df.fillna(0, inplace=True)
print(f"✓ Data preprocessing complete")

Running on Google Colab
✗ Error mounting Google Drive: mount failed
  Falling back to local dataset search...
✗ Error mounting Google Drive: mount failed
  Falling back to local dataset search...


NameError: name 'csv_files' is not defined

In [ ]:
import os
import glob

# Auto-detect and find CSV files
print("="*60)
print("SETUP: Searching for dataset files...")
print("="*60)

possible_paths = [
    '../Dataset',           # Standard location
    '../../Dataset',        # Parent directory
    '../dataset',           # Lowercase
    '../../dataset',
    'Dataset',              # Current directory
    'dataset',
]

dataset_dir = None
for path in possible_paths:
    if os.path.exists(path) and os.path.isdir(path):
        csv_count = len(glob.glob(os.path.join(path, '*.csv')))
        if csv_count > 0:
            dataset_dir = os.path.abspath(path)
            print(f"✓ Found {csv_count} CSV files in: {dataset_dir}")
            break

if dataset_dir is None:
    print("\n✗ ERROR: Cannot find Dataset folder!")
    print("\nExpected structure:")
    print("  Project/")
    print("  ├── Code/")
    print("  │   └── Quantum_CNN_for_Intrusion_Detection_On_MIot_dataset.ipynb")
    print("  └── Dataset/")
    print("      ├── Attack.csv")
    print("      ├── environmentMonitoring.csv")
    print("      └── patientMonitoring.csv")
    print("\nPlease ensure your CSV files are in the Dataset folder!")
    raise FileNotFoundError("Dataset folder not found. Please check file structure.")

# List all CSV files found
print(f"\nCSV files found in {dataset_dir}:")
csv_files_found = glob.glob(os.path.join(dataset_dir, '*.csv'))
for i, file in enumerate(csv_files_found, 1):
    file_size = os.path.getsize(file) / (1024*1024)  # Convert to MB
    print(f"  {i}. {os.path.basename(file)} ({file_size:.2f} MB)")

print("\n" + "="*60)

In [ ]:
# Scale numerical features
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=2)
principal_components = pca.fit_transform(df_scaled)
principal_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

# Plot PCA Results
plt.figure(figsize=(8, 6))
plt.scatter(principal_df['PC1'], principal_df['PC2'])
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Result: First Two Principal Components')
plt.grid(True)
plt.show()

print(f"Number of features pre-PCA: {df.shape[1]}")
print(f"Number of features post-PCA: {principal_df.shape[1]}")

In [ ]:
# Split data into training and testing sets
try:
    X_train, X_test, y_train, y_test = train_test_split(df_scaled, y, test_size=0.3, random_state=42)
    print(f"✓ Data split successfully")
    print(f"  X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
except Exception as e:
    print(f"✗ Error splitting data: {e}")
    raise

# Convert to numpy arrays to avoid pandas indexing issues
X_train = np.asarray(X_train)
X_test = np.asarray(X_test)
y_train = np.asarray(y_train)
y_test = np.asarray(y_test)
print(f"✓ Converted to numpy arrays")

# Identify categorical columns for encoding (from original df)
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Apply Frequency Encoding to categorical features (if any exist)
if len(categorical_cols) > 0:
    print(f"\nProcessing {len(categorical_cols)} categorical columns: {categorical_cols}")
    for col in categorical_cols:
        try:
            frequency_map = pd.Series(X_train[:, df.columns.get_loc(col)]).value_counts().to_dict()
            X_train_col_freq = pd.Series(X_train[:, df.columns.get_loc(col)]).map(frequency_map).fillna(0).values
            X_test_col_freq = pd.Series(X_test[:, df.columns.get_loc(col)]).map(frequency_map).fillna(0).values
            print(f"  ✓ Encoded {col}")
        except Exception as e:
            print(f"  ⚠ Warning: Could not encode {col}: {e}")
else:
    print("✓ No categorical columns found - skipping frequency encoding")

# Final scaling
try:
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    print(f"✓ Final scaling complete")
except Exception as e:
    print(f"✗ Error during scaling: {e}")
    raise

print(f"\n✓ Data preprocessing pipeline complete")
print(f"  Final X_train shape: {X_train.shape}")
print(f"  Final X_test shape: {X_test.shape}")
print(f"  Classes in y_train: {np.unique(y_train)}")
print(f"  Classes in y_test: {np.unique(y_test)}")

In [ ]:
import subprocess
import sys

def install_package(package_name):
    """Safely install a package with error handling"""
    try:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        print(f"✓ {package_name} installed successfully")
        return True
    except Exception as e:
        print(f"✗ Failed to install {package_name}: {e}")
        return False

# Install required packages
install_package("qiskit-aer")

In [ ]:
# Install quantum and deep learning packages
packages = [
    "qiskit",
    "qiskit-machine-learning",
    "torch torchvision torchaudio"
]

for package in packages:
    install_package(package)

print("\n✓ All quantum and deep learning packages installed")

In [ ]:
# Verify quantum computing libraries
try:
    import qiskit
    import qiskit_machine_learning
    print("✓ Qiskit version:", qiskit.__version__)
    print("✓ Qiskit Machine Learning version:", qiskit_machine_learning.__version__)
except ImportError as e:
    print(f"✗ Error importing quantum libraries: {e}")
    raise

try:
    import torch
    print(f"✓ PyTorch version: {torch.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except ImportError as e:
    print(f"✗ Error importing PyTorch: {e}")
    raise

In [ ]:
import torch
import torch.nn as nn
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.primitives import Sampler

# Define Quantum Circuit with feature map and ansatz
n_qubits = 8  # Set number of qubits
feature_map = ZZFeatureMap(n_qubits)
ansatz = RealAmplitudes(n_qubits, reps=1)
qc = QuantumCircuit(n_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

# Use QNN with Sampler backend
sampler = Sampler()
qnn = SamplerQNN(circuit=qc, input_params=feature_map.parameters, weight_params=ansatz.parameters, sampler=sampler)

# Connect Quantum Model with PyTorch
quantum_model = TorchConnector(qnn)

In [ ]:
import torch
import torch.nn as nn
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.primitives import Sampler

# Step 1: Define Quantum Circuit with feature map and ansatz
n_qubits = 8
feature_map = ZZFeatureMap(n_qubits)  # Encodes classical data into quantum states
ansatz = RealAmplitudes(n_qubits, reps=1)  # Quantum trainable layer

# Combine feature map and ansatz into a single circuit
qc = QuantumCircuit(n_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

# Step 2: Define QNN using Qiskit Sampler
sampler = Sampler()
qnn = SamplerQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    sampler=sampler
)

# Step 3: Convert QNN to PyTorch model using TorchConnector
quantum_model = TorchConnector(qnn)

# Print model summary to check compatibility
print("Quantum CNN Model initialized successfully! ")
print(quantum_model)

In [ ]:
install_package("torchmetrics")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import torchmetrics
from torch.nn import CrossEntropyLoss
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import time
from torch.optim import AdamW
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Verify data is available
try:
    assert X_train.shape[0] > 0 and X_test.shape[0] > 0, "Empty training or test data"
    assert len(y_train) > 0 and len(y_test) > 0, "Empty labels"
    print(f"✓ Data verified: X_train {X_train.shape}, X_test {X_test.shape}")
except Exception as e:
    print(f"✗ Data validation error: {e}")
    raise

# Define CNN model
class ClassicalCNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ClassicalCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Define Quantum CNN Model
class QuantumCNN(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super(QuantumCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(64, 256, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(256, 64)
        self.dropout = nn.Dropout(0.5)
        self.out = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.global_avg_pool(x).squeeze(-1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)  # logits

# Prepare data
try:
    n_qubits = min(8, X_train.shape[1])  # Use up to 8 qubits or available features
    num_classes = len(np.unique(y_train))
    
    print(f"✓ Model parameters: n_qubits={n_qubits}, num_classes={num_classes}")
    
    X_train_qcnn_t = torch.tensor(X_train[:, :n_qubits], dtype=torch.float32).to(device)
    X_test_qcnn_t = torch.tensor(X_test[:, :n_qubits], dtype=torch.float32).to(device)
    X_train_cnn_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_cnn_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    
    print(f"✓ Data converted to tensors")
    print(f"  X_train_qcnn_t: {X_train_qcnn_t.shape}")
    print(f"  X_train_cnn_t: {X_train_cnn_t.shape}")
except Exception as e:
    print(f"✗ Error preparing data: {e}")
    raise

# Instantiate models
try:
    classical_model = ClassicalCNN(input_dim=X_train.shape[1], num_classes=num_classes).to(device)
    quantum_model = QuantumCNN(input_dim=n_qubits, num_classes=num_classes).to(device)
    print(f"✓ Models instantiated successfully")
except Exception as e:
    print(f"✗ Error creating models: {e}")
    raise

# Optimizer & Loss
optimizer_cnn = Adam(classical_model.parameters(), lr=0.001)
optimizer_qcnn = AdamW(quantum_model.parameters(), lr=0.0005, weight_decay=1e-4)
loss_fn = CrossEntropyLoss()

# Training loop with error handling
epochs = 50
metrics = {
    'cnn': {'a. loss': [], 'b. accuracy': [], 'c. precision': [], 'd. recall': [], 'e. time': []},
    'qcnn': {'a. loss': [], 'b. accuracy': [], 'c. precision': [], 'd. recall': [], 'e. time': []}
}

print(f"\n{'='*60}")
print(f"Starting training for {epochs} epochs...")
print(f"{'='*60}\n")

for epoch in range(epochs):
    try:
        for name, model, optimizer, X_input in zip(
            ['qcnn', 'cnn'],
            [quantum_model, classical_model],
            [optimizer_qcnn, optimizer_cnn],
            [X_train_qcnn_t, X_train_cnn_t]
        ):
            model.train()
            optimizer.zero_grad()
            start_time = time.time()

            try:
                output = model(X_input)
                loss = loss_fn(output, y_train_t)
                
                if torch.isnan(loss):
                    print(f"✗ NaN loss detected at epoch {epoch} for {name}")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                pred = output.argmax(dim=1)
                acc = (pred == y_train_t).float().mean().item()
                
                try:
                    prec = torchmetrics.functional.precision(
                        pred, y_train_t, average='macro', 
                        num_classes=num_classes, task="multiclass"
                    )
                    rec = torchmetrics.functional.recall(
                        pred, y_train_t, average='macro',
                        num_classes=num_classes, task="multiclass"
                    )
                except Exception as e:
                    prec = torch.tensor(0.0)
                    rec = torch.tensor(0.0)

                end_time = time.time()
                elapsed_time = end_time - start_time

                metrics[name]['a. loss'].append(loss.item())
                metrics[name]['b. accuracy'].append(acc)
                metrics[name]['c. precision'].append(prec.item() if isinstance(prec, torch.Tensor) else prec)
                metrics[name]['d. recall'].append(rec.item() if isinstance(rec, torch.Tensor) else rec)
                metrics[name]['e. time'].append(elapsed_time)

                if epoch % 10 == 0:
                    print(f"Epoch {epoch:3d} | {name.upper()} Loss: {loss.item():.4f} Acc: {acc:.4f}")
            
            except Exception as e:
                print(f"✗ Training error for {name} at epoch {epoch}: {e}")
                continue
    
    except KeyboardInterrupt:
        print(f"\n⚠ Training interrupted at epoch {epoch}")
        break
    except Exception as e:
        print(f"✗ Epoch {epoch} failed: {e}")
        continue

print(f"\n{'='*60}")
print("✓ Training complete!")
print(f"{'='*60}\n")

# Plotting Curves with error handling
try:
    plt.figure(figsize=(16, 10))
    for i, metric in enumerate(['a. loss', 'b. accuracy', 'c. precision', 'd. recall', 'e. time']):
        plt.subplot(3, 2, i+1)
        if len(metrics['qcnn'][metric]) > 0 and len(metrics['cnn'][metric]) > 0:
            plt.plot(metrics['qcnn'][metric], label='QCNN', linestyle='--', color='blue')
            plt.plot(metrics['cnn'][metric], label='CNN', linestyle='-', color='orange')
        plt.title(f'{metric.replace("a. ", "").replace("b. ", "").replace("c. ", "").replace("d. ", "").replace("e. ", "").capitalize()} Over Epochs')
        plt.xlabel('Epoch')
        plt.ylabel(metric.capitalize())
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.show()
    print("✓ Training curves plotted successfully")
except Exception as e:
    print(f"✗ Error plotting training curves: {e}")

In [ ]:
# Evaluate on test set with comprehensive error handling
try:
    print("Evaluating models on test set...")
    
    with torch.no_grad():
        quantum_model.eval()
        classical_model.eval()
        
        y_pred_qcnn = quantum_model(X_test_qcnn_t).detach().cpu().numpy()
        y_pred_cnn = classical_model(X_test_cnn_t).detach().cpu().numpy()
    
    y_pred_qcnn_labels = np.argmax(y_pred_qcnn, axis=1)
    y_pred_cnn_labels = np.argmax(y_pred_cnn, axis=1)
    
    print("✓ Predictions generated successfully\n")
    
    # Classification Report
    print("="*60)
    print("CLASSIFICATION REPORTS")
    print("="*60)
    
    for name, y_pred in zip(["QCNN", "CNN"], [y_pred_qcnn_labels, y_pred_cnn_labels]):
        try:
            print(f"\n{name} Performance:")
            print("-"*60)
            print(classification_report(y_test, y_pred))
        except Exception as e:
            print(f"✗ Error generating classification report for {name}: {e}")
    
    # Confusion Matrices
    print("\n" + "="*60)
    print("CONFUSION MATRICES")
    print("="*60 + "\n")
    
    for name, y_pred in zip(["QCNN", "CNN"], [y_pred_qcnn_labels, y_pred_cnn_labels]):
        try:
            plt.figure(figsize=(5, 4))
            cm = confusion_matrix(y_test, y_pred)
            ConfusionMatrixDisplay(cm).plot(cmap='YlGnBu')
            plt.title(f'{name} Confusion Matrix')
            plt.grid(False)
            plt.show()
        except Exception as e:
            print(f"✗ Error plotting confusion matrix for {name}: {e}")
    
    # ROC Curve (Binary classification only)
    if num_classes == 2:
        try:
            print("\n" + "="*60)
            print("ROC CURVE (Binary Classification)")
            print("="*60 + "\n")
            
            plt.figure(figsize=(6, 5))
            
            fpr_qcnn, tpr_qcnn, _ = roc_curve(y_test, y_pred_qcnn[:, 1])
            roc_auc_qcnn = auc(fpr_qcnn, tpr_qcnn)
            plt.plot(fpr_qcnn, tpr_qcnn, label=f"QCNN (AUC = {roc_auc_qcnn:.3f})", color='blue', linewidth=2)

            fpr_cnn, tpr_cnn, _ = roc_curve(y_test, y_pred_cnn[:, 1])
            roc_auc_cnn = auc(fpr_cnn, tpr_cnn)
            plt.plot(fpr_cnn, tpr_cnn, label=f"CNN (AUC = {roc_auc_cnn:.3f})", color='orange', linewidth=2)

            plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')
            plt.xlabel('False Positive Rate', fontsize=12)
            plt.ylabel('True Positive Rate', fontsize=12)
            plt.title('ROC Curve Comparison', fontsize=14)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.show()
            
            print(f"QCNN AUC: {roc_auc_qcnn:.4f}")
            print(f"CNN AUC: {roc_auc_cnn:.4f}")
        except Exception as e:
            print(f"✗ Error plotting ROC curve: {e}")
    else:
        print(f"⚠ Multiclass classification detected ({num_classes} classes)")
        print("  ROC curves are primarily for binary classification")
    
    print("\n" + "="*60)
    print("✓ Evaluation complete!")
    print("="*60)

except Exception as e:
    print(f"✗ Fatal error during evaluation: {e}")
    import traceback
    traceback.print_exc()
    raise